In [3]:
import importlib
import hdf_utils
from pathlib import Path
import os
from tqdm.auto import tqdm
import datetime
import sys
import h5py
sys.path.append('/Users/michaellavender/Documents/BUSPC/batch_processing')
from binary_to_hdf import whole_binary_to_hdf

# Conversion

In [ ]:
from histutils.convert.__main__ import convert_DMC_to_hdf5
dmcfn = Path('/Volumes/HST-1/2013-03-24/2013-03-24T05-55-CamSer7196.DMCdata')
outfile = Path('/Volumes/TOSHIBA EXT/HDF/2013-03-24/2013-03-24T05-55-CamSer7196.h5')
outfile.parent.mkdir(parents=True, exist_ok=True)


In [ ]:
# nmea and xml same name as hdf
convert_DMC_to_hdf5(dmcfn, outfile, {"header_bytes": 4})


In [ ]:
# nmea and xml different name as hdf
convert_DMC_to_hdf5(dmcfn, outfile, {"header_bytes": 4, 'nmeaFile': '/Volumes/4TB #1 HST1/2014-04-17ext/2014-04-17T07-00-CamSer7196.nmea', 'xmlFile': '/Volumes/4TB #1 HST1/2014-04-17ext/2014-04-17T07-00-CamSer7196.xml'})

# Rough Keogram

In [ ]:
hdf_fn = '/Volumes/TOSHIBA EXT/HDF/2013-04-14/1387/2013-04-14-CamSer1387.h5'
out_dir = '/Volumes/TOSHIBA EXT/HDF/2013-04-14/1387/videos + keogram_1hz_binsize10'
hdf_utils.make_keogram_rougher(hdf_path=hdf_fn, out_dir=out_dir, bin_size_seconds=60, sample_interval=3600)`

# Batch rough keograms

In [ ]:
import os
hdf_parent_folder = '/Volumes/TOSHIBA EXT/HDF'
skip_dirs = {"HDF Tests", "4tb_drive1"}
files = []

# Get files
for root, dirs, filenames in os.walk(hdf_parent_folder):
    dirs[:] = [d for d in dirs if d not in skip_dirs]
    for name in filenames:
        if name.endswith(".h5"):
            files.append(os.path.join(root, name))

files = [Path(f) for f in files]

In [ ]:
# Process
done = []
errors = []
for file in tqdm(files):
    print(file.name)
    out_dir = file.parent
    try:
        hdf_utils.make_keogram_rougher(hdf_path=file, out_dir=out_dir, bin_size_seconds=60, sample_interval=3600)
        done.append(file)
    except Exception as e:
        errors.append((file, e))
        print(f"  failed: {e}")

# Hourly Videos & Keograms

In [4]:
import keogram_consumer_hourly as kch
import keogram_consumer
import stats_consumer
import video_consumer
importlib.reload(kch)
importlib.reload(keogram_consumer)
importlib.reload(stats_consumer)
importlib.reload(video_consumer)
importlib.reload(hdf_utils)

<module 'hdf_utils' from '/Users/michaellavender/Documents/BUSPC/processing/hdf_utils.py'>

In [ ]:
# Setup
start = datetime.datetime(2013, 4, 14, 7, 40, 0, tzinfo=datetime.timezone.utc)
start_unix = start.timestamp() # needed for norm

hdf_fn = Path('/Volumes/TOSHIBA EXT/HDF/2013-04-14/1387/2013-04-14-CamSer1387.h5')
out_dir = Path('/Volumes/TOSHIBA EXT/HDF/2013-04-14/1387/hourly_linear')
out_dir.mkdir(exist_ok=True, parents=True)

In [ ]:
# Get norm
f = h5py.File(hdf_fn, 'r')
imgs = f['rawimg']
ut = f['ut1_unix']
start_idx = hdf_utils.find_closest_item(ut, start_unix)
norm = hdf_utils.compute_norm(imgs, ut, 1, start_idx = start_idx, low_percentile=0.1, high_percentile=99.9, linear=True)

In [ ]:
hdf_utils.make_hourly_videos_keograms(hdf_path=hdf_fn, out_dir=out_dir, bin_size=5, playback_speed=15, output_hz=1, norm=norm)

# Helper Funcitions

In [ ]:
import datetime
import h5py
def get_time_range(hdf_fn):
    with h5py.File(hdf_fn, 'r') as f:
        ut = f['ut1_unix']
        start_epoch, end_epoch = ut[0], ut[-1]
        start = datetime.datetime.fromtimestamp(start_epoch, tz=datetime.timezone.utc)
        end = datetime.datetime.fromtimestamp(end_epoch, tz=datetime.timezone.utc)

    print(start.strftime("%Y-%m-%d %H:%M:%S"), end.strftime("%Y-%m-%d %H:%M:%S"))
    return start, end